In [ ]:
import pandas as pd

df = pd.concat([pd.read_csv('training_data/products.txt', sep= '\t', names=["text","label"]), \
			   pd.read_csv('training_data/negatifs.txt', sep= '\t', names=["text","label"])])


In [ ]:
df.head()

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def process_data(row):
    text = row['text']
    text = str(text)
    text = ' '.join(text.split())

    encodings = tokenizer(text, padding="max_length", truncation=True, max_length=128)

    label = 0
    if row['label'] == 'product':
        label += 1

    encodings['label'] = label
    encodings['text'] = text

    return encodings

In [ ]:
processed_data = []

for i in range(len(df[:11000])):
    processed_data.append(process_data(df.iloc[i]))

In [ ]:
from sklearn.model_selection import train_test_split

new_df = pd.DataFrame(processed_data)

train_df, valid_df = train_test_split(
    new_df,
    test_size=0.19,
    random_state=2022
)

In [ ]:
import pyarrow as pa
from datasets import Dataset

train_hg = Dataset(pa.Table.from_pandas(train_df))
valid_hg = Dataset(pa.Table.from_pandas(valid_df))

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

In [ ]:

import torch
from transformers import TrainingArguments, Trainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model.to(device)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',  # Specify GPU device
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hg,
    eval_dataset=valid_hg,
    tokenizer=tokenizer
)

In [ ]:
trainer.train()


In [ ]:
trainer.evaluate()

In [ ]:
from transformers import AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

new_model = AutoModelForSequenceClassification.from_pretrained('./model/').to(device)

In [ ]:

from transformers import AutoTokenizer

new_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def get_prediction(tokenizer, model, text):
	device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
	encoding = tokenizer(text, return_tensors='pt', truncation=True)
	encoding = {k: v.to(device) for k, v in encoding.items()}

	outputs = model(**encoding)

	logits = outputs.logits
	probs = torch.nn.functional.softmax(logits, dim=-1)
	sigmoid = torch.nn.Sigmoid()
	probs = sigmoid(logits.squeeze().detach())
	probs = probs.detach().numpy()
	if np.argmax(probs, axis=-1) == 1:
		return text
		

In [ ]:
class crawler:
	def __init__(self, urls):
		self.urls = urls
		self.tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
		self.model = AutoModelForSequenceClassification.from_pretrained('./model/').to(device)

	def get_prediction(self, tokenizer, model, text):
		device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		encoding = tokenizer(text, return_tensors='pt', truncation=True)
		encoding = {k: v.to(device) for k, v in encoding.items()}

		outputs = model(**encoding)

		logits = outputs.logits
		probs = torch.nn.functional.softmax(logits, dim=-1)
		sigmoid = torch.nn.Sigmoid()
		probs = sigmoid(logits.squeeze().detach())
		probs = probs.detach().numpy()
		if np.argmax(probs, axis=-1) == 1:
			return text

	def crawl(self):
		products = []
		driver = webdriver.Chrome(ChromeDriverManager().install())
		for url in self.urls:
			print(f"Crawling {url}...")
			try:
				driver.get(url)
				# Get the page source
				page_source = driver.page_source
				soup = bs(page_source, 'html.parser')
				links = soup.find_all('a')
				headers = soup.find_all(['h1','h2','h3','h4','h5','h6'])
				driver.quit()

				for header in headers:
					if self.get_prediction(self.tokenizer, self.model, header.get_text()) is not None:
						products.append(header.get_text())

				for link in links:
					heads = link.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
					if heads is None:
						texts = link.get_text(separator=' | ').split(' | ')
						for text in texts:
							if self.get_prediction(self.tokenizer, self.model, text) is not None:
								products.append(text)
								break
				print(" DONE\n")
			except Exception as e:
				print(f"Exception occurred while crawling: {str(e)}")
				continue
		driver.quit()

		return products
		
	